##Creating a table of relevant MSDS variables linked to CCU063 maternity interpreter cohort 

Purpose - to link MSDS charging with maternity interpreter cohort

Authors - Majel McGranahan supported by Lars Murdock

Reviewed - Not reviewed, needs cleaning up by MM

#0 Parameters

In [0]:
%run "./CCU063_03-D01-parameters"

# 1 Load Maternity Interpreter Cohort Table

In [0]:
maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_maternity_interpreter_folicacid')

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (maternity_interpreter_cohort
.select('person_id_mother_deid', 'uniqpregid', 'est_preg_start',  'lookback_start', 'lookback_issue_flag', 'NHS_NUMBER_interpreter', 'SNOMED_conceptId', 'SNOMED_conceptId_description', 'DATE_interpreter', 'RECORD_DATE_interpreter', 'person_id_demo', 'Dob', 'eth5', 'region', 'imd_quintile', 'imd_decile', 'in_gdppr', 'gdppr_min_date', 'interpreter_use', 'record_before_lookback', 'ageatbookingmother', 'delivery_date', 'agefinal', 'folicacid')
        )



In [0]:
count_var(maternity_interpreter_cohort, 'uniqpregid')

# 2 Load MSDS table



In [0]:
msds_demo= (spark.table(f'{dbc_old}.msds_v2_demographics_booking_and_pregnancy_all_years_archive')
          #.filter(F.col('ADMIDATE') > "2018-01-01")
          .filter(f.col('archived_on') == tmp_archived_on)
          )

In [0]:
msds_demo.count()

In [0]:
#msds_demo1 = (
#    msds_demo.where(f.col('OvsVisChCat').isNotNull())
#)

#msds_demo1.count()

In [0]:
msds_demo= (
    msds_demo
    .select(f.col('person_id_mother_deid').alias('person_id_mother_msds'), 
            f.col('uniqpregid').alias('uniquepregid_msds'),
            f.col('ovsvischcat').alias('ovsvischcat'),
            f.col('ovsvischcatappdate').alias('ovsvischcatappdate'),
           #f.col('langcode').alias('langcode'),
           #f.col('complexsocialfactorsind').alias('complexsocialfactors'),
           #f.col('previouslivebirths').alias('previouslivebirths'),
           #f.col('previousstillbirths').alias('previousstillbirths'),
           #f.col('previouslosseslessthan24weeks').alias('previouslosseslessthan24weeks'),
          #f.col('folicacidsupplement').alias('folicacid1'),
          # f.col('gestagebooking').alias('gestagebooking'),
          f.col('archived_on').alias('archived_on'),
          #           .distinct() 
           ).dropDuplicates()
)

In [0]:
msds_demo.count()

In [0]:
display(msds_demo)

In [0]:
msds_demo_charging =    (
msds_demo
.select(f.col('person_id_mother_msds').alias('person_id_mother_msds'), 
            f.col('uniquepregid_msds').alias('uniquepregid_msds'),
            f.col('archived_on').alias('archived_on'),
            f.col('ovsvischcatappdate').alias('ovsvischcatappdate'),
          f.col('ovsvischcat').alias('ovsvischcat1')))



In [0]:
msds_demo_charging = (
    msds_demo_charging.where(f.col('ovsvischcat1').isNotNull())
    .distinct() 
    .dropDuplicates()
)

msds_demo_charging.count()


In [0]:
tab(msds_demo_charging, 'ovsvischcat1')

##Convert 00s, ZZs etc to 'null' and drop nulls

In [0]:
msds_demo_charging=msds_demo_charging.withColumn('ovsvischcat', 
                                                                       f.when(f.col('ovsvischcat1') == 'A', 'A')
                                                                       .when(f.col('ovsvischcat1') == 'B', 'B')
                                                                       .when(f.col('ovsvischcat1') == 'C', 'C')
                                                                       .when(f.col('ovsvischcat1') == 'D', 'D')
                                                                       .when(f.col('ovsvischcat1') == 'E', 'E')
                                                                       .when(f.col('ovsvischcat1') == 'F', 'F')
                                                                       .when(f.col('ovsvischcat1') == 'P', 'P')
                                                                       .otherwise(None))

In [0]:
tab(msds_demo_charging, 'ovsvischcat')

In [0]:
msds_demo_charging=msds_demo_charging.filter(f.col('ovsvischcat').isNotNull())


In [0]:
tab(msds_demo_charging, 'ovsvischcat')

In [0]:
#from https://sparkbyexamples.com/pyspark/pyspark-select-first-row-of-each-group/
#Will this order from smallest to largest DATE_interpreter?

from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number
w2 = Window.partitionBy("uniquepregid_msds").orderBy(col("archived_on"))
msds_demo_charging = ( msds_demo_charging
                          .withColumn("row",row_number().over(w2))
                          .filter(col("row") == 1)) 

In [0]:
msds_demo_charging = msds_demo_charging.drop('row')

In [0]:
msds_demo_charging = msds_demo_charging.drop('ovsvischcat1')

In [0]:
##check
count_var(msds_demo_charging, 'person_id_mother_msds')

In [0]:
##check
count_var(msds_demo_charging, 'uniquepregid_msds')

In [0]:
display(msds_demo_charging)

#3 Join MSDS to Maternity interpreter cohort table 

In [0]:
##Attempting left join based on https://www.geeksforgeeks.org/pyspark-join-types-join-two-dataframes/

##check row count in each table before and after join (row count in output table will be same as left table row count - filtered lookup)

# left join on two dataframes 
maternity_interpreter_charging=maternity_interpreter_cohort.join(msds_demo_charging, 
               maternity_interpreter_cohort.uniqpregid == msds_demo_charging.uniquepregid_msds,  
               "left")




#display table
display(maternity_interpreter_charging)
display(maternity_interpreter_charging.printSchema())

In [0]:
tab(maternity_interpreter_charging, 'ovsvischcat')

In [0]:
tab(maternity_interpreter_charging, 'ovsvischcat', 'interpreter_use')

#4 Save table with charging

In [0]:
outName = f'{proj}_maternity_interpreter_charging'

# save
maternity_interpreter_charging.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

In [0]:
maternity_interpreter_charging = spark.table(f'{dbc}.{proj}_maternity_interpreter_charging')